# CNC Guard — Entrenamiento y evaluación

Este notebook documenta la limpieza del CSV, el desequilibrio de clases, el feature engineering, la evaluación con métricas orientadas a Recall y el guardado del artefacto que utiliza `scoring-api`.

El objetivo de negocio es reducir falsos negativos: una parada no anticipada puede costar 1.200 €/hora.

## 1. Preparación y carga de datos

Se utiliza exclusivamente `ai4i2020_train.csv`; los datos del simulador no se mezclan con el entrenamiento.

In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'scoring'))
from train_model import (
    DATA_PATH, ENGINEERED_FEATURE_COLUMNS, FEATURE_COLUMNS,
    QUALITY_COLUMN, TARGET_COLUMN, add_engineered_features,
    clean_training_data, choose_failure_threshold,
)

raw = pd.read_csv(ROOT / 'data' / 'ai4i2020_train.csv')
df = clean_training_data(raw)
print(f'Filas iniciales: {len(raw):,}')
print(f'Filas limpias: {len(df):,}')
df.head()

## 2. EDA y desequilibrio de clases

La clase de fallo es minoritaria. Por eso Accuracy no es suficiente: un modelo que siempre predijera 0 tendría una accuracy alta, pero Recall igual a 0.

In [ ]:
display(df.describe(include='all').T)
print('Distribución de machine_type:')
display(df[QUALITY_COLUMN].value_counts())
print('Distribución del objetivo:')
display(df[TARGET_COLUMN].value_counts())
print(f'Tasa de fallo: {df[TARGET_COLUMN].mean():.2%}')

df[TARGET_COLUMN].value_counts().sort_index().plot(kind='bar', color=['#4caf50', '#e53935'])
plt.title('Distribución de fallos')
plt.xlabel('machine_failure')
plt.ylabel('Número de registros')
plt.show()

## 3. Limpieza y feature engineering

`temp_diff` representa la diferencia entre temperatura de proceso y ambiente. `power_est` aproxima la carga mecánica como `rpm × torque_Nm`. Estas variables tienen interpretación física y se calculan igual en el API durante la inferencia.

In [ ]:
df = add_engineered_features(df)
display(df[[*FEATURE_COLUMNS, *ENGINEERED_FEATURE_COLUMNS, TARGET_COLUMN]].head())
display(df[[*ENGINEERED_FEATURE_COLUMNS, TARGET_COLUMN]].groupby(TARGET_COLUMN).mean())

## 4. División sin data leakage

La partición es estratificada para conservar la proporción de fallos. El modelo no ve los datos del conjunto de validación durante el entrenamiento.

In [ ]:
from sklearn.model_selection import train_test_split

NUMERIC_FEATURES = [*FEATURE_COLUMNS, *ENGINEERED_FEATURE_COLUMNS]
MODEL_FEATURES = [*NUMERIC_FEATURES, QUALITY_COLUMN]
X = df[MODEL_FEATURES]
y = df[TARGET_COLUMN].astype(int)
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Train: {len(X_train):,} filas; validación: {len(X_val):,} filas')
print(f'Fallos en train: {y_train.mean():.2%}; validación: {y_val.mean():.2%}')

## 5. Comparación de modelos

Se compara un Random Forest supervisado con una regresión logística balanceada. El modelo elegido para producción es Random Forest porque captura relaciones no lineales entre sensores y permite obtener probabilidades. `class_weight='balanced'` da más importancia a los fallos minoritarios.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, recall_score, precision_score, f1_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

def preprocessor():
    return ColumnTransformer([
        ('numeric', 'passthrough', NUMERIC_FEATURES),
        ('quality', OneHotEncoder(handle_unknown='ignore'), [QUALITY_COLUMN]),
    ])

models = {
    'Random Forest': Pipeline([
        ('preprocessor', preprocessor()),
        ('classifier', RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)),
    ]),
    'Logistic Regression': Pipeline([
        ('preprocessor', preprocessor()),
        ('classifier', LogisticRegression(class_weight='balanced', max_iter=2000, random_state=42)),
    ]),
}

comparison = []
for name, model in models.items():
    model.fit(X_train, y_train)
    probabilities = model.predict_proba(X_val)[:, 1]
    predictions = probabilities >= 0.5
    comparison.append({
        'modelo': name,
        'ROC-AUC': roc_auc_score(y_val, probabilities),
        'Recall': recall_score(y_val, predictions, zero_division=0),
        'Precision': precision_score(y_val, predictions, zero_division=0),
        'F1': f1_score(y_val, predictions, zero_division=0),
    })
pd.DataFrame(comparison).sort_values('Recall', ascending=False)

## 6. Umbral orientado a Recall y matriz de confusión

Se selecciona un umbral que alcance Recall objetivo de 0,76. Esto refleja que no detectar un fallo es más costoso que revisar una alerta falsa.

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, classification_report, confusion_matrix

rf = models['Random Forest']
rf_probabilities = rf.predict_proba(X_val)[:, 1]
threshold, threshold_info = choose_failure_threshold(y_val, rf_probabilities, target_recall=0.76)
rf_predictions = rf_probabilities >= threshold
print(f'Umbral seleccionado: {threshold:.4f}')
print(threshold_info)
print(classification_report(y_val, rf_predictions, zero_division=0))

ConfusionMatrixDisplay(
    confusion_matrix(y_val, rf_predictions),
    display_labels=['No fallo', 'Fallo'],
).plot(cmap='Blues')
plt.title('Matriz de confusión — Random Forest con umbral ajustado')
plt.show()

## 7. Importancia de variables

La importancia se interpreta junto con las métricas: no demuestra causalidad, pero ayuda a explicar qué señales utiliza el modelo.

In [ ]:
classifier = rf.named_steps['classifier']
pre = rf.named_steps['preprocessor']
names = [*NUMERIC_FEATURES, *pre.named_transformers_['quality'].get_feature_names_out([QUALITY_COLUMN])]
importance = pd.Series(classifier.feature_importances_, index=names).sort_values()
importance.plot(kind='barh', figsize=(8, 5), color='#1976d2')
plt.title('Importancia de variables — Random Forest')
plt.xlabel('Importancia')
plt.show()
importance.sort_values(ascending=False)

## 8. Entrenamiento final y guardado

El script de producción entrena además los clasificadores de tipo de fallo y el Isolation Forest, guarda el umbral elegido y genera `models/cnc_failure_model.pkl`.

In [ ]:
from train_model import main as train_production_artifact
train_production_artifact()
print('Artefacto guardado en:', ROOT / 'models' / 'cnc_failure_model.pkl')